# ERGT Colab Phase 3 Short Proof Baseline

Run this notebook on Google Colab with `Runtime > Change runtime type > GPU`.

This is a one-hour-oriented calibration run. It keeps the proof baseline architecture but uses `max_steps=1000` instead of `10000`.

This is not Gate 1 proof. It answers: how fast is the proof-sized baseline on the current Colab GPU, and does the stronger baseline train cleanly?

In [ ]:
REPO_URL = "https://github.com/jalaljafari2009/ERGT.git"
BRANCH = "main"
PROJECT_DIR = "/content/ERGT"

CONFIG_PATH = "configs/baseline/short_proof_wikitext2.json"
DATA_DIR = "data/processed/wikitext2_gpt2_ctx256"
RESULTS_PATH = "runs/phase0_baseline/short_proof_wikitext2/baseline_results.json"
RUN_DIR = "runs/phase0_baseline/short_proof_wikitext2"

DEVICE = "cuda"
RUN_BASELINE = True
FORCE_RERUN = False

## 1. Runtime probe

In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

print("cwd:", os.getcwd())
print("python:", sys.version)
print("platform:", platform.platform())
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))

nvidia_smi = shutil.which("nvidia-smi")
print("nvidia-smi:", nvidia_smi)
if nvidia_smi:
    subprocess.run([nvidia_smi], check=False)

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Set Colab hardware accelerator to GPU.")
print("cuda_device_name:", torch.cuda.get_device_name(0))

## 2. Clone/update repo and install dependencies

In [ ]:
project_path = Path(PROJECT_DIR)
if project_path.exists():
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
repo_head = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
)
print("repo:", repo_head.strip())
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,data,viz]"],
    check=True,
)

## 3. Prepare WikiText-2

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--tokenizer",
        "gpt2",
        "--context-length",
        "256",
    ],
    cwd=PROJECT_DIR,
    check=True,
)

## 4. Run short proof baseline

In [ ]:
def run_command(command):
    print("\n$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)


results_path = Path(PROJECT_DIR) / RESULTS_PATH

if RUN_BASELINE:
    if results_path.exists() and not FORCE_RERUN:
        print("Skipping run because results already exist:", results_path)
    else:
        run_command(
            [
                sys.executable,
                "experiments/train_baseline.py",
                "--config",
                CONFIG_PATH,
                "--data-dir",
                DATA_DIR,
                "--device",
                DEVICE,
            ]
        )
else:
    print("RUN_BASELINE is False; no training executed.")

if not results_path.exists():
    raise FileNotFoundError(f"Missing result: {results_path}")

## 5. Print result and full-proof estimate

In [ ]:
with results_path.open("r", encoding="utf-8") as handle:
    result = json.load(handle)

print(json.dumps(result, indent=2, sort_keys=True))

tokens = float(result["total_training_tokens"])
seconds = float(result["total_wall_clock_seconds"])
tokens_per_second = float(result["average_tokens_per_second"])
full_tokens = 10000 * 16 * 256
estimated_full_seconds = full_tokens / max(tokens_per_second, 1e-9)

print("\n=== Estimate ===")
print("short_run_tokens:", int(tokens))
print("short_run_seconds:", round(seconds, 2))
print("tokens_per_second:", round(tokens_per_second, 2))
print("estimated_full_proof_hours:", round(estimated_full_seconds / 3600, 2))

## 6. Archive light outputs

In [ ]:
light_root = Path("/content/ergt_short_proof_baseline_light")
if light_root.exists():
    shutil.rmtree(light_root)

source_run_dir = Path(PROJECT_DIR) / RUN_DIR
target_run_dir = light_root / RUN_DIR
target_run_dir.mkdir(parents=True, exist_ok=True)

for path in source_run_dir.rglob("*"):
    if not path.is_file() or path.suffix not in {".json", ".jsonl"}:
        continue
    relative = path.relative_to(source_run_dir)
    destination = target_run_dir / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, destination)
    print("included:", destination.relative_to(light_root))

light_archive = shutil.make_archive(
    "/content/ergt_short_proof_baseline_light", "zip", light_root
)
print("Light archive ready:", light_archive)